In [9]:
import pandas as pd
import os
import glob

def read_tower_data(base_path, file_pattern='Pinguim_b*', min_count=50):
    """
    Read and process tower data from multiple years and files.

    Parameters
    ----------
    base_path : str
        Base directory containing yearly folders
    years : list of str
        Years to read (e.g., ['2021', '2022'])
    file_pattern : str
        Pattern to match files (default: 'CSV_58812.tm*')
    min_count : int
        Minimum number of records per hour for QC filtering

    Returns
    -------
    df_hourly : pandas.DataFrame
        Hourly resampled DataFrame with QC applied
    """

    # ======================
    # 1. Read all files
    # ======================
    folder_path = os.path.join(base_path)
    files = glob.glob(os.path.join(folder_path, file_pattern))

    file_list = sorted(files)

    # Column names
    #var_name = ['year', 'doy', 'hour', 'non', 't2m', 'rh', 'pres',
    #            'SUp', 'SDn', 'ws', 'wd', 'Tsol', 'vol']

    # Column names
    var_name = ['id', 'year', 'doy', 'hour', 'sec', 'SUp', 'SDn', 'LUp', 'LDn', 'CNR4TC', 'CNR4TK', 'RsNet', 'RlNet', 'Albedo', 'RN', 
                 'LUpCo', 'LDnCo', 'tair', 'rh', 'press', 'SHF']

    df_list = []

    for file in file_list:
        df_tmp = pd.read_csv(file, header=None)
        df_tmp.columns = var_name
        df_list.append(df_tmp)

    df = pd.concat(df_list, ignore_index=True)

    # ======================
    # 2. Create datetime
    # ======================
    df['hh'] = df['hour'] // 100
    df['mm'] = df['hour'] % 100

    df['datetime'] = (
        pd.to_datetime(df['year'], format='%Y') +
        pd.to_timedelta(df['doy'] - 1, unit='D') +
        pd.to_timedelta(df['hh'], unit='h') +
        pd.to_timedelta(df['mm'], unit='m')
    )

    df = df.drop(columns=['hh', 'mm'])

    # ======================
    # 3. Set index & clean
    # ======================
    df = df.set_index('datetime')
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]

    # ======================
    # 4. Hourly resampling + QC
    # ======================
    df_mean  = df.resample('1h').mean()
    df_count = df.resample('1h').count()

    df_hourly = df_mean.copy()
    df_hourly[df_count < min_count] = pd.NA

    return df_hourly

def read_tower_data_w(base_path, file_pattern='Pinguim_w*', min_count=50):
    """
    Read and process tower data from multiple years and files.

    Parameters
    ----------
    base_path : str
        Base directory containing yearly folders
    years : list of str
        Years to read (e.g., ['2021', '2022'])
    file_pattern : str
        Pattern to match files (default: 'CSV_58812.tm*')
    min_count : int
        Minimum number of records per hour for QC filtering

    Returns
    -------
    df_hourly : pandas.DataFrame
        Hourly resampled DataFrame with QC applied
    """

    # ======================
    # 1. Read all files
    # ======================
    folder_path = os.path.join(base_path)
    files = glob.glob(os.path.join(folder_path, file_pattern))

    file_list = sorted(files)

    # Column names
    #var_name = ['year', 'doy', 'hour', 'non', 't2m', 'rh', 'pres',
    #            'SUp', 'SDn', 'ws', 'wd', 'Tsol', 'vol']

    # Column names
    var_name = ['id', 'year', 'doy', 'hour', 'sec', 'ws', 'wd']

    df_list = []

    for file in file_list:
        df_tmp = pd.read_csv(file, header=None)
        df_tmp.columns = var_name
        df_list.append(df_tmp)

    df = pd.concat(df_list, ignore_index=True)

    # ======================
    # 2. Create datetime
    # ======================
    df['hh'] = df['hour'] // 100
    df['mm'] = df['hour'] % 100

    df['datetime'] = (
        pd.to_datetime(df['year'], format='%Y') +
        pd.to_timedelta(df['doy'] - 1, unit='D') +
        pd.to_timedelta(df['hh'], unit='h') +
        pd.to_timedelta(df['mm'], unit='m')
    )

    df = df.drop(columns=['hh', 'mm'])

    # ======================
    # 3. Set index & clean
    # ======================
    df = df.set_index('datetime')
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]

    # ======================
    # 4. Hourly resampling + QC
    # ======================
    df_mean  = df.resample('1h').mean()
    df_count = df.resample('1h').count()

    df_hourly = df_mean.copy()
    df_hourly[df_count < min_count] = pd.NA

    return df_hourly


In [10]:
base_path = "/home/lacrio/cryoextremes-practicals/data/low_freq/"

df_hourly = read_tower_data(base_path)

df_hourly_ws = read_tower_data_w(base_path)

In [11]:
df_met_ws = pd.merge(
    df_hourly, df_hourly_ws,
    left_index=True,
    right_index=True,
    how="inner"
)



In [12]:
file_path = "/home/lacrio/cryoextremes-practicals/data/pross/met_data_1h.csv"
vars = ['SUp', 'SDn', 'LUpCo', 'LDnCo', 'tair', 'rh', 'press', 'ws', 'wd', 'SHF']
df_met_ws[vars].to_csv(file_path)